In [1]:
# Run these small tutorials on CPU; select a GPU before importing JAX for larger runs.
import os
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np
import marlax as mx
from marlax.envs import CooperativeGrid, GridState
from marlax.models import QTable
from marlax.methods import QLearning, SARSA, ExpectedSARSA


In [2]:
# 1. A model represents one table. It contains no exploration or learning rule.
env = CooperativeGrid()
model = QTable(initial_value=0.5)
table = model.init(jax.random.PRNGKey(0), env.spec.num_observations, env.spec.num_actions)
values, carry = model.apply(table, jnp.array(0), model.initial_carry(), None)
{"table_shape": table.shape, "initial_q": np.asarray(values), "carry": carry}


{'table_shape': (4375, 5),
 'initial_q': array([0.5, 0.5, 0.5, 0.5, 0.5], dtype=float32),
 'carry': ()}

In [3]:
# 2. An agent routes one table per animal, or a single shared table.
independent = mx.Agent(model, sharing="independent")
shared = mx.Agent(model, sharing="shared")
independent_q = independent.init(jax.random.PRNGKey(1), env.spec)
shared_q = shared.init(jax.random.PRNGKey(1), env.spec)
{"independent": independent_q.shape, "shared": shared_q.shape,
 "independent_slots": np.asarray(independent.slots(2)), "shared_slots": np.asarray(shared.slots(2))}


{'independent': (2, 4375, 5),
 'shared': (1, 4375, 5),
 'independent_slots': array([0, 1], dtype=int32),
 'shared_slots': array([0, 0], dtype=int32)}

In [4]:
# 3. Masks exclude unavailable actions even when their raw Q-value is largest.
from marlax.agents import action_probabilities
inputs = mx.PolicyInput(jnp.array([0, 0]), jnp.array([[True, True, False, True, True]] * 2))
q = independent_q.at[:, 0, :].set(jnp.array([1.0, 3.0, 100.0, 0.0, 0.0]))
values = independent.values(q, inputs)
probabilities = action_probabilities(values, inputs.action_mask, epsilon=0.2)
assert jnp.all(probabilities[:, 2] == 0)
{"legal_epsilon_greedy_probabilities": np.asarray(probabilities)}


{'legal_epsilon_greedy_probabilities': array([[0.05, 0.85, 0.  , 0.05, 0.05],
        [0.05, 0.85, 0.  , 0.05, 0.05]], dtype=float32)}

In [5]:
# 4. Agent.act samples behavior; a frozen policy acts greedily without updating q.
decision = independent.act(q, inputs, (), jax.random.PRNGKey(2), exploration=0.2)
policy = mx.FrozenPolicy(independent, q)
greedy = policy.act(inputs, policy.initial_carry(), None)
assert jnp.array_equal(greedy.actions, jnp.array([1, 1]))
{"sampled_actions": np.asarray(decision.actions), "frozen_actions": np.asarray(greedy.actions)}


{'sampled_actions': array([1, 0], dtype=int32),
 'frozen_actions': array([1, 1], dtype=int32)}

In [6]:
# 5. Exact greedy ties choose the lowest legal index.
# Shared weights do not turn joint observations into local observations.
tied = mx.FrozenPolicy(shared, shared_q)
assert jnp.array_equal(tied.act(inputs, (), None).actions, jnp.array([0, 0]))
{"tie_actions": np.asarray(tied.act(inputs, (), None).actions)}


{'tie_actions': array([0, 0], dtype=int32)}